# SongSeeker Running Example

This example demonstrates how to use the SongSeeker library to search for songs based on text queries.

First, we set up the working directory as the project root:

In [2]:
import os
import sys
from pathlib import Path
import ipynbname

# Set working directory to project root and add src to sys.path
nb_path = ipynbname.path() 
parent_dir = nb_path.parent.parent
os.chdir(parent_dir)

if os.path.abspath("./src") not in sys.path:
    sys.path.append(os.path.abspath("./src"))

print("Working directory:", os.getcwd())
print("sys.path:", sys.path)

Working directory: c:\Users\jiach\Repositories\SongSeeker
sys.path: ['c:\\Users\\jiach\\miniconda3\\envs\\SongSeeker\\python312.zip', 'c:\\Users\\jiach\\miniconda3\\envs\\SongSeeker\\DLLs', 'c:\\Users\\jiach\\miniconda3\\envs\\SongSeeker\\Lib', 'c:\\Users\\jiach\\miniconda3\\envs\\SongSeeker', '', 'c:\\Users\\jiach\\miniconda3\\envs\\SongSeeker\\Lib\\site-packages', 'c:\\Users\\jiach\\miniconda3\\envs\\SongSeeker\\Lib\\site-packages\\win32', 'c:\\Users\\jiach\\miniconda3\\envs\\SongSeeker\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\jiach\\miniconda3\\envs\\SongSeeker\\Lib\\site-packages\\Pythonwin', 'c:\\Users\\jiach\\Repositories\\SongSeeker\\src']


## Preprocessing and Loading the Data

Due to the limited hardware resources, we will limit our search to the a subset of 10,000 rows and the Learn to Rank training dataset with a subset of 5000 rows.

These two subsets have been preprocessed already and saved at `sample_data\processed\genius-clean-with-title-artist-10000.csv` and `sample_data\processed\genius-clean-with-title-artist-5000.csv` by `python ./src/preprocessing.py`.

In [29]:
import pandas as pd
data_path_search = "sample_data/processed/genius-clean-with-title-artist-10000.csv"
dataset = pd.read_csv(data_path_search)

## Initializing BM25 Searcher

In [30]:
import search_bm25

bm25 = search_bm25.TextRetrieval()
if not bm25.load_cache("cache/bm25"):
    print("Preprocessing the dataset...")
    bm25.processed_docs = bm25.preprocess_docs(dataset['lyrics'])

    print("Building vocabulary and document-term matrix...")
    bm25.build_vocabulary()
    bm25.build_doc_term_matrix()
    bm25.save_cache("cache/bm25")
            
else:
    print("BM25 cache loaded.")

Cache found. Loading from cache/bm25...                        
3 items loaded from cache.
BM25 cache loaded.


## Initializing Word2Vec Searcher

In [33]:
import search_w2v

w2v = search_w2v.TextRetrieval()
if not w2v.load_cache("cache/w2v"):
    print("Preprocessing the dataset...")
    w2v.read_and_preprocess_genius(data_path_search)
    
    print("Loading Word2Vec embeddings...")
    w2v.load_embeddings()
    w2v.build_doc_W2V_cache(max_doc_tokens=200, keep_full_mats=False)
    w2v.save_cache("cache/w2v")
    
else:
    print("Word2Vec cache loaded.")

[W2V] Loading cache from cache/w2v...
10000 documents loaded from cache.
Word2Vec cache loaded.


## Initializing BERT Searcher

In [34]:
import search_bert

bert = search_bert.SongBiEncoderSearcher()
if not bert.load_cache("cache/bert"):
    print("Building BERT embeddings...")
    bert.build_from_csv(data_path_search)
    bert.save_cache("cache/bert")
    
else:
    print("BERT cache loaded.")

Model running on device: cuda:0
Loading cache from cache/bert...
10000 embeddings loaded from cache.
BERT cache loaded.


# Train Learn to Rank Model

In [35]:

!python -m src.learn_to_rank

Loading full dataset from: c:\Users\jiach\Repositories\SongSeeker\sample_data\processed\Labeled_genius-clean-with-title-artist-5000.csv
Splitting data: Using first 4000 rows for TRAINING.
Saved temp training file to: c:\Users\jiach\Repositories\SongSeeker\sample_data\processed\temp_train_subset.csv
--- 1. Data Loading (Training Subset) ---
Reading file: c:\Users\jiach\Repositories\SongSeeker\sample_data\processed\temp_train_subset.csv
Using text column: 'lyrics'

--- 2. Initializing BM25 ---
Processing document 1/4000
Processing document 2/4000
Processing document 3/4000
Processing document 4/4000
Processing document 5/4000
Processing document 6/4000
Processing document 7/4000
Processing document 8/4000
Processing document 9/4000
Processing document 10/4000
Processing document 11/4000
Processing document 12/4000
Processing document 13/4000
Processing document 14/4000
Processing document 15/4000
Processing document 16/4000
Processing document 17/4000
Processing document 18/4000
Processi

# The Real SongSeeker

In [36]:
import SongSeeker
top_k= 10
ranker = SongSeeker.LearnedRanker(csv_path=data_path_search)
while True:
    try:
        q = input("\nQuery> ").strip()
        if not q:
            print("Empty query, exiting.")
            break

        result = ranker.rank(q, top_k=top_k)
        # pretty-print top results
        with pd.option_context("display.max_colwidth", 80):
            print(result)

    except (EOFError, KeyboardInterrupt):
        print("\nExiting.")
        break

[Init] Loading corpus from: sample_data/processed/genius-clean-with-title-artist-10000.csv
[Init] Using text column: 'lyrics'
[Init] Building BM25 cache...
Cache found. Loading from cache/bm25...                        
3 items loaded from cache.
[Init] BM25 cache loaded.
[Init] Building Word2Vec cache (cosine mode only)...
[W2V] Loading cache from cache/w2v...
10000 documents loaded from cache.
[Init] W2V cache loaded.
[Init] Initializing BERT bi-encoder...
Model running on device: cuda:0
Loading cache from cache/bert...
10000 embeddings loaded from cache.
[Init] BERT cache loaded.
[Init] Loading model from: models\logreg_model.pkl
[Init] Loading scaler from: models\scaler.pkl
[Init] LearnedRanker ready.
Empty query, exiting.
